# CROCUS Data on ESS-DIVE

**ESS-DIVE** — the Environmental System Science Data Infrastructure for a Virtual Ecosystem — is a U.S. Department of Energy data repository for Earth and environmental science data. It is used to archive, share, discover, and cite observational, experimental, and modeling datasets from DOE-supported environmental research, including data from ecological, hydrological, geochemical, and urban systems.

The **CROCUS** project (Community Research on Climate and Urban Science) publishes its urban sensor-network data here. This notebook shows how to find those datasets, see what files they contain, and download a dataset's daily files and stitch them into a single NetCDF archive — all without needing to log in.

NetCDF is a common scientific data format for array-oriented data, especially time series, gridded environmental data, and model output. In Python, `xarray` provides a convenient way to open NetCDF files as labeled datasets.

## Browsing the data by hand first

Before writing any code, it helps to see what we're working with. Visit **https://data.ess-dive.lbl.gov/data** and type `CROCUS` in the **Search** box. You'll get a list of datasets. Each one is a *package* with its own **DOI** (a permanent identifier you can cite), and each package holds many individual data files.

The rest of this notebook does the same search programmatically so we can automate the download.

## 1. Setup

Import the libraries we'll use. `requests` talks to the ESS-DIVE web API, and `xarray` reads and combines NetCDF files.

In [ ]:
import json
import os
from datetime import datetime, timezone
#from urllib.parse import quote

import pandas as pd
import requests
import xarray as xr

Set up where things live. `BASE` is the ESS-DIVE API address. We keep two folders:

- **`datge/essdive/`** holds the finished, combined archives (the files you'll keep).
- **`data/essdive/tmp/`** holds the raw daily files we download along the way. These are disposable — once an archive is built you can delete `data/essdive/tmp/` to reclaim space.

No login or password is needed: the CROCUS datasets we download here are public.

In [ ]:
BASE = "https://api.ess-dive.lbl.gov/packages"

OUTDIR  = "data/essdive"            # final concatenated .nc
WORKDIR = "data/essdive/tmp"        # disposable daily chunks

os.makedirs(WORKDIR, exist_ok=True)
os.makedirs(OUTDIR, exist_ok=True)

## 2. Search for CROCUS datasets

`search_crocus` asks the ESS-DIVE API for public datasets whose metadata mentions `CROCUS`. The API returns results one page at a time, so the loop below keeps requesting pages until it has collected them all.

Two quirks of this API worth knowing:
- `rowStart` is **1-indexed** — the first result is row 1, not row 0. (Passing 0 returns an error.)
- `isPublic="true"` restricts results to datasets anyone can download.

In [ ]:
def search_crocus(text="CROCUS", row_start=1, page_size=25):
    """One page of search results from the ESS-DIVE packages API."""
    params = {"text": text, "isPublic": "true",
              "rowStart": row_start, "pageSize": page_size}
    r = requests.get(BASE, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

# Page through all results
results, start, page_size = [], 1, 25
while True:
    page = search_crocus(row_start=start, page_size=page_size)
    results.extend(page["result"])
    if start + page_size > page["total"]:     # rowStart is 1-indexed
        break
    start += page_size

print(f"Found {page['total']} CROCUS datasets")

## 3. Build a manifest of every dataset and its files

The search above only returns *summary* information (title, DOI). To see the actual **files** inside each dataset, we fetch each package individually with `fetch_package`. `build_manifest` does this for every dataset, then caches the result to `essdive_manifest.json` so re-running the notebook doesn't repeatedly hit the API.

For each dataset we record three counts, because a package mixes several kinds of file:
- **`n_entries`** — everything listed, including the metadata XML and PNG quicklook images.
- **`n_files`** — entries that actually have a download link.
- **`n_nc`** — just the NetCDF (`.nc`) data files, which is what we archive.

> **Tip:** if you edit `build_manifest`, delete `essdive_manifest.json` before re-running — otherwise it loads the old cached copy instead of rebuilding.

In [ ]:
OBJECT_BASE = "https://data.ess-dive.lbl.gov/catalog/d1/mn/v2/object/"

def fetch_package(doi):
    """Full package JSON (metadata + file list) for one DOI."""
    r = requests.get(f"{BASE}/{doi}", timeout=60)
    r.raise_for_status()
    return r.json()

def file_url(f):
    """Direct download URL for a distribution entry.
    Older ESS-DIVE packages put the URL in contentUrl. Newer ones omit it
    and give only an identifier; the URL is OBJECT_BASE + identifier
    (the same pattern the older packages stored in contentUrl)."""
    if f.get("contentUrl"):
        return f["contentUrl"]
    ident = f.get("identifier")
    return f"{OBJECT_BASE}{ident}" if ident else None

def build_manifest(search_results, cache="essdive_manifest.json"):
    """Fetch every package once, cache to disk, and return a list of
    dicts with the DOI, title, file counts, and downloadable file list."""
    if os.path.exists(cache):
        with open(cache) as f:
            return json.load(f)
    manifest = []
    for i, rec in enumerate(search_results, 1):
        doi = rec["id"]
        pkg = fetch_package(doi)["dataset"]
        files = pkg.get("distribution", []) or []

        dl_files = []
        for f in files:
            url = file_url(f)
            if url:                              # keep entries we can download
                dl_files.append({"name": f.get("name"),
                                 "url": url,
                                 "size": f.get("contentSize"),
                                 "format": f.get("encodingFormat")})
        n_nc = sum(1 for f in dl_files
                   if f["name"] and f["name"].endswith(".nc"))
        manifest.append({
            "doi": doi,
            "title": pkg.get("name"),
            "viewUrl": rec.get("viewUrl"),
            "n_entries": len(files),             # raw distribution count (incl. xml/png)
            "n_files": len(dl_files),            # downloadable files
            "n_nc": n_nc,                        # NetCDF files only
            "files": dl_files,
        })
        print(f"[{i}/{len(search_results)}] {pkg.get('name')[:60]}... "
              f"{n_nc} .nc / {len(dl_files)} files")
    with open(cache, "w") as f:
        json.dump(manifest, f, indent=2)
    return manifest

def find_idx(manifest, *keywords):
    """Return the index of the first dataset whose title contains
    all keywords (case-insensitive). Raises if not found."""
    for i, m in enumerate(manifest):
        title = (m["title"] or "").lower()
        if all(k.lower() in title for k in keywords):
            return i
    raise ValueError(f"No dataset matching {keywords}")

In [ ]:
import os
if os.path.exists("essdive_manifest.json"):
    os.remove("essdive_manifest.json")
manifest = build_manifest(results)     # re-fetches with the new file_url logic

### A note on datasets that show `0 files`

Some datasets print `0 .nc / 0 files`. This does **not** mean they're empty — it means their data is hosted externally (ESS-DIVE's "Tier 2" storage, reached through Globus) rather than served directly by this API. The `download_and_archive` function below only works on datasets whose files have a direct download link. The Tier 2 datasets can still be downloaded by hand from their landing page (the `viewUrl` in the manifest), or via Globus.

So the manifest is your map of *what this notebook can pull directly* versus *what needs the website*.

## 4. Summary: how much data is here?

This totals the downloadable files and their size. **ESS-DIVE reports file sizes in kilobytes (KB)**, so we divide the summed size by 1,000,000 to get gigabytes. (A common mistake is to treat the numbers as bytes, which makes the total look ~1000× too small.)

In [ ]:
# Summary before downloading anything
total_nc = sum(m["n_nc"] for m in manifest)
total_files = sum(m["n_files"] for m in manifest)
total_kb = sum(f["size"] or 0 for m in manifest for f in m["files"])
print(f"{len(manifest)} datasets, {total_nc} NetCDF files "
      f"({total_files} downloadable files total), "
      f"{total_kb/1e6:.2f} GB")

## 5. Browse the datasets as a table

This turns the manifest into a sortable table so you can decide what to download *before* pulling any data. The **`idx`** column is the number you pass to the download function in the next step. Sizes are converted from KB to MB here.

In [ ]:
rows = []
for i, m in enumerate(manifest):
    mb = sum(f["size"] or 0 for f in m["files"]) / 1000   # KB -> MB
    rows.append({"idx": i, "title": m["title"][:55],
                 "nc_files": m["n_nc"], "size_MB": round(mb, 1)})

df = pd.DataFrame(rows).sort_values("size_MB", ascending=False)
pd.set_option("display.max_colwidth", 60)
df[["idx", "title", "nc_files", "size_MB"]]

## 6. Download one dataset and build a combined archive

Each CROCUS dataset stores its data as many small NetCDF files — one for every few days. `download_and_archive` downloads all of a dataset's `.nc` files into `data/essdive/tmp/`, then uses `xarray.open_mfdataset` to stitch them together along the time axis into one file named `<label>_essdive.nc`.

Two features worth pointing out in the code:
- **Resumable downloads.** Each file is written to a temporary `.part` name and only renamed once it's complete, so an interrupted download never leaves a half-written file behind. Re-running skips files already on disk.
- **Provenance.** The combined file records where it came from — the DOI, the landing-page URL, and the date you downloaded it — in its global attributes, so the archive stays self-documenting even if it's copied somewhere else.

Pass the `idx` from the table above and a short `label` (used in the output filename).

In [ ]:
def _read_one(path):
    """Open a single ESS-DIVE NetCDF, capturing its variable attributes.
    open_mfdataset with conflicting per-file attrs silently drops them, so we
    keep a copy from the first file and re-assert it after concatenation."""
    return xr.open_dataset(path)


def download_and_archive(idx, label, manifest):
    """Download all .nc files for one dataset, concat along time, and write
    {label}_essdive.nc with provenance attrs. Variable-level metadata
    (units, standard_name, ...) from the source files is preserved."""
    entry = manifest[idx]
    doi = entry["doi"]
    ncfiles = [f for f in entry["files"] if f["name"].endswith(".nc")]
    print(f"{entry['title'][:55]}: {len(ncfiles)} netCDF files")

    # download each (skip existing, atomic)
    local = []
    sub = os.path.join(WORKDIR, label)
    os.makedirs(sub, exist_ok=True)
    for f in ncfiles:
        dest = os.path.join(sub, f["name"])
        if not (os.path.exists(dest) and os.path.getsize(dest) > 0):
            with requests.get(f["url"], stream=True, timeout=120) as r:
                r.raise_for_status()
                tmp = dest + ".part"
                with open(tmp, "wb") as out:
                    for chunk in r.iter_content(1 << 16):
                        out.write(chunk)
                os.replace(tmp, dest)
        local.append(dest)

    files = sorted(local)

    # Capture per-variable attributes from the first file BEFORE concat.
    # (open_mfdataset drops variable attrs that are not identical across files.)
    with _read_one(files[0]) as first:
        var_attrs = {name: dict(da.attrs) for name, da in first.variables.items()}

    # concat along time; combine_attrs="override" keeps the GLOBAL attrs of the
    # first file instead of dropping conflicting ones.
    ds = xr.open_mfdataset(
        files,
        combine="nested",
        concat_dim="time",
        combine_attrs="override",
    )

    # Re-assert variable-level attributes that the concat may have dropped.
    for name, attrs in var_attrs.items():
        if name in ds.variables:
            for k, v in attrs.items():
                ds[name].attrs.setdefault(k, v)

    # provenance (global)
    ds.attrs["source_archive"] = f"ESS-DIVE {doi}"
    ds.attrs["source_url"] = entry["viewUrl"]
    ds.attrs["essdive_retrieved"] = datetime.now(timezone.utc).isoformat()
    ds.attrs["essdive_n_files"] = len(ncfiles)

    out = os.path.join(OUTDIR, f"{label}_essdive.nc")
    ds.to_netcdf(out)
    ds.close()
    print(f"wrote {out}  ({os.path.getsize(out)/1e6:.1f} MB)")

    # quick confirmation that units survived the round trip
    check = xr.open_dataset(out)
    missing = [v for v in check.data_vars if "units" not in check[v].attrs]
    if missing:
        print(f"  NOTE: {len(missing)} variable(s) still lack units: {missing}")
    else:
        print("  all data variables carry a units attribute")
    check.close()
    return out

### Example: the NEIU rooftop weather and air-quality datasets

Run the cell below to build two archives — the Northeastern Illinois University rooftop weather station (WXT) and air-quality sensor (AQT). Confirm the `idx` values still point at the NEIU datasets in the table above first; they can shift if ESS-DIVE adds new packages.

In [ ]:
wxt_idx = find_idx(manifest, "Argonne National Lab", "Weather Data")
aqt_idx = find_idx(manifest, "Argonne National Lab", "Air Quality Data", "Prairie")

print(wxt_idx, manifest[wxt_idx]["title"])
print(aqt_idx, manifest[aqt_idx]["title"])


In [ ]:

download_and_archive(wxt_idx, "ANL_wxt", manifest)
download_and_archive(aqt_idx, "ANL_aqt", manifest)

## 7. Open and inspect a finished archive

Once an archive is written, open it back up to check what's inside — the variables, the time range, and the provenance attributes we attached. Change the filename to inspect a different archive.

In [ ]:
ds = xr.open_dataset(os.path.join(OUTDIR, "NEIU_wxt_essdive.nc"))
ds

## 8. Metadata sanity check

Whether or not a variable *has* a `units` string, the string can still be wrong. This check cross-examines each variable three ways: does it carry `units` and `standard_name` at all; do the units make physical sense for the declared `standard_name` (e.g. a `wind_speed` must not be in `celsius`); and do the actual data ranges look consistent with those units. It flags the specific defects seen in the CROCUS WXT archive — wind speed mislabeled as `celsius`, undocumented variables, and rainfall with no accumulation/rate semantics.


In [ ]:
import numpy as np

# Physical sanity: which units are plausible for a given CF standard_name.
# Keyed by standard_name; value is a set of acceptable unit strings (lowercased).
_PLAUSIBLE = {
    "air_temperature":       {"celsius", "degc", "k", "kelvin"},
    "dew_point_temperature": {"celsius", "degc", "k", "kelvin"},
    "relative_humidity":     {"percent", "%", "1"},
    "air_pressure":          {"hpa", "pa", "mbar", "mb"},
    "wind_speed":            {"m s-1", "m/s", "meters per second"},
    "wind_from_direction":   {"degree", "degrees", "degree_true"},
    "precipitation_amount":  {"kg m-2", "mm"},
}

# Rough physical range each standard_name should fall in (after unit interpretation).
_RANGE = {
    "air_temperature":       (-60, 60),
    "dew_point_temperature": (-60, 50),
    "relative_humidity":     (0, 100),
    "air_pressure":          (800, 1100),   # hPa
    "wind_speed":            (0, 120),       # m/s
    "wind_from_direction":   (0, 360),
}


def check_metadata(path):
    """Report per-variable metadata quality and flag likely defects."""
    ds = xr.open_dataset(path)
    print(f"== {path} ==")
    print(f"   conventions: {ds.attrs.get('conventions', '(none)')}")
    print(f"   {len(ds.data_vars)} data variables, "
          f"{ds.sizes.get('time', '?')} time steps\n")

    problems = []
    header = f"   {'variable':16s} {'units':12s} {'standard_name':24s} status"
    print(header)
    print("   " + "-" * (len(header) - 3))

    for name in ds.data_vars:
        da = ds[name]
        units = da.attrs.get("units")
        sname = da.attrs.get("standard_name")
        flags = []

        if units is None:
            flags.append("NO UNITS")
        if sname is None and "long_name" not in da.attrs:
            flags.append("no standard_name/long_name")

        # units-vs-standard_name plausibility
        if sname and units is not None:
            ok = _PLAUSIBLE.get(sname)
            if ok and units.strip().lower() not in ok:
                flags.append(f"UNITS '{units}' inconsistent with '{sname}'")

        # data-range plausibility against the standard_name
        if sname in _RANGE:
            vals = np.asarray(da.values, dtype="float64")
            vals = vals[np.isfinite(vals)]
            if vals.size:
                lo, hi = _RANGE[sname]
                if vals.min() < lo or vals.max() > hi:
                    flags.append(
                        f"range [{vals.min():.1f}, {vals.max():.1f}] "
                        f"outside expected [{lo}, {hi}]")

        # accumulation/rate ambiguity for precipitation
        if sname == "precipitation_amount" and "long_name" not in da.attrs:
            flags.append("no long_name: accumulation vs rate unstated")

        status = "OK" if not flags else "  ||  ".join(flags)
        print(f"   {name:16s} {str(units):12.12s} {str(sname):24.24s} {status}")
        if flags:
            problems.append((name, flags))

    print()
    if problems:
        print(f"   {len(problems)} variable(s) with metadata issues:")
        for name, flags in problems:
            for f in flags:
                print(f"     - {name}: {f}")
    else:
        print("   no metadata issues detected.")
    ds.close()
    return problems


# Run it on the finished archive (and works on raw per-file downloads too).
_ = check_metadata(os.path.join(OUTDIR, "NEIU_wxt_essdive.nc"))


In [ ]:
import xarray as xr, glob
aqt_files = sorted(glob.glob("data/essdive/tmp/UIC_aqt/*.nc"))
print(len(aqt_files), "files\n")

a = xr.open_dataset(aqt_files[0])
b = xr.open_dataset(aqt_files[-1])

print("=== file 0 ===")
print("dims:", dict(a.sizes))
print("coords:", list(a.coords))
print("data_vars:", list(a.data_vars))
print("\n=== last file ===")
print("dims:", dict(b.sizes))
print("coords:", list(b.coords))

# compare the dp size across all files
print("\n=== dp size per file ===")
for f in aqt_files:
    d = xr.open_dataset(f)
    print(f"  {f.split('/')[-1]}: dp={d.sizes.get('dp')}, time={d.sizes.get('time')}")
    d.close()

## 9. Network-wide metadata audit (raw source files)

Section 8 checks one *finished* archive. This section inspects the **raw per-file downloads** in `data/essdive/tmp/<label>/` — the files exactly as ESS-DIVE published them, before any concatenation of ours. It answers a different question: *what metadata defects exist in the source archive, and are they network-wide or specific to one site?*

For each download folder it:

- runs the same per-variable checks as `check_metadata`, but across every file;
- **aggregates** each defect with a count (e.g. `wind_mean_10s units='celsius' — 97/97 files`) instead of printing one block per file;
- flags **intra-site inconsistency** — cases where files in the *same* folder disagree on a variable's attributes. That inconsistency is what caused the attribute loss in our own concatenation, so it is worth seeing explicitly.

It is read-only and safe to re-run as more sites are downloaded.


In [ ]:
import glob
from collections import defaultdict


def _var_signature(da):
    """A hashable summary of one variable's metadata, for spotting files that
    disagree. (units, standard_name, has_long_name)."""
    return (da.attrs.get("units"),
            da.attrs.get("standard_name"),
            "long_name" in da.attrs)


def _defects_for_var(name, da, vals_minmax):
    """Return a list of defect strings for one variable (no data load here;
    ranges are passed in)."""
    units = da.attrs.get("units")
    sname = da.attrs.get("standard_name")
    flags = []
    if units is None:
        flags.append("no units")
    if sname is None and "long_name" not in da.attrs:
        flags.append("no standard_name/long_name")
    if sname and units is not None:
        ok = _PLAUSIBLE.get(sname)
        if ok and units.strip().lower() not in ok:
            flags.append(f"units '{units}' inconsistent with '{sname}'")
    if sname == "precipitation_amount" and "long_name" not in da.attrs:
        flags.append("rainfall: accumulation vs rate unstated")
    if sname in _RANGE and vals_minmax is not None:
        lo, hi = _RANGE[sname]
        vmin, vmax = vals_minmax
        if vmin < lo or vmax > hi:
            flags.append(f"data range [{vmin:.1f}, {vmax:.1f}] outside [{lo}, {hi}]")
    return flags


def audit_site(folder, check_ranges=True):
    """Audit every .nc file in one raw download folder. Aggregates defects with
    counts and flags intra-site attribute inconsistency."""
    files = sorted(glob.glob(os.path.join(folder, "*.nc")))
    label = os.path.basename(folder.rstrip("/"))
    if not files:
        print(f"[{label}] no .nc files")
        return None

    n = len(files)
    defect_counts = defaultdict(int)            # (var, defect) -> count of files
    signatures = defaultdict(set)               # var -> set of metadata signatures
    var_seen = defaultdict(int)                 # var -> count of files containing it

    for path in files:
        ds = xr.open_dataset(path)
        for name in ds.data_vars:
            da = ds[name]
            var_seen[name] += 1
            signatures[name].add(_var_signature(da))
            vmm = None
            if check_ranges and da.attrs.get("standard_name") in _RANGE:
                v = np.asarray(da.values, dtype="float64")
                v = v[np.isfinite(v)]
                if v.size:
                    vmm = (float(v.min()), float(v.max()))
            for flag in _defects_for_var(name, da, vmm):
                defect_counts[(name, flag)] += 1
        ds.close()

    print(f"== {label}: {n} file(s) ==")

    # Aggregated defects
    if defect_counts:
        print("  defects (count / files containing the variable):")
        for (var, flag), c in sorted(defect_counts.items()):
            print(f"    {var:16s} {flag:48s} {c}/{var_seen[var]}")
    else:
        print("  no defects detected.")

    # Intra-site inconsistency: a variable whose metadata signature is not the
    # same in every file that contains it.
    inconsistent = {v: sigs for v, sigs in signatures.items() if len(sigs) > 1}
    if inconsistent:
        print("  INTRA-SITE INCONSISTENCY (files disagree on these variables' metadata):")
        for v, sigs in sorted(inconsistent.items()):
            print(f"    {v}: {len(sigs)} distinct (units, standard_name, has_long_name) signatures")
            for s in sorted(sigs, key=lambda x: tuple("" if e is None else str(e) for e in x)):
                print(f"        units={s[0]!r}, standard_name={s[1]!r}, long_name={s[2]}")
    print()
    return {"label": label, "n_files": n,
            "defects": dict(defect_counts), "inconsistent": inconsistent}


def audit_all(tmp_root=WORKDIR, check_ranges=True):
    """Audit every site folder under the raw-download root."""
    folders = sorted(p for p in glob.glob(os.path.join(tmp_root, "*"))
                     if os.path.isdir(p))
    if not folders:
        print(f"no site folders under {tmp_root!r} — download something first.")
        return []
    print(f"Auditing {len(folders)} site folder(s) under {tmp_root!r}\n")
    return [audit_site(f, check_ranges=check_ranges) for f in folders]


# Run across everything downloaded so far.
_audit = audit_all()
